# 🧪 Analíticas de sangre (tabular) — **v1 LIGERO** (CPU · modelo independiente)
## TFM · Módulo Tabular (Laboratorio) · Universidad de Salamanca

---

## 🎯 Qué es esta versión (la más ligera de las 3)
La señal tabular tiene techo (~0.67–0.69 AUC). Esta versión es la **línea base rápida**: un **único XGBoost**
one-vs-rest que aprovecha el **manejo nativo de NaN** de los árboles (no hace falta imputar) y unas pocas
**variables clínicas derivadas**.

| Las 3 versiones | Modelo | Coste CPU |
|---|---|---|
| **v1 (esta, ligera)** | **un XGBoost** one-vs-rest tuneado (ligero) | **minutos** |
| v2 (óptima) | ensemble XGB+LightGBM+LogReg+MLP con tuning por modelo | ~1 h |
| v3 (pesada) | v2 + TabPFN (nube) + stacking de 2º nivel de los tabulares | ~horas |

## 🔄 Adaptación a las conclusiones del EDA
- **Etiquetado FINAL**: `POS=(==1)` · `NEG=(==0)|(NaN→0)` · **−1 ENMASCARADO** (U-Ignore).
  **Se elimina** la derivación de negativos desde *No Finding*, que introducía **sesgo de espectro**. → §3
- **Métrica primaria = AUC-PR** (`macro_AP_path`) con **IC bootstrap**; AUC-ROC secundaria. → §3 y §1
- **Raw en árboles** (NaN nativo, escala clínica); nunca raw+percentil a la vez (r>0,90). → §9
- **Flags de missingness** (MNAR): los positivos tienen menos ausencias. → §6
- **Ratios clínicos**: BUN/Creatinina, Neutrófilos/Linfocitos, RDW×Edad. → §5/§10
- `scale_pos_weight` por etiqueta (desbalanceo 1,7:1–6,3:1, **nunca SMOTE**). → §3
- **Clusters fisiológicos**: NO se reducen aquí (los árboles son robustos a la colinealidad); la
  reducción por representante se explora en v2/v3, que sí tienen modelos densos. → §8

## 🛡️ Bases del proyecto (intactas)
Enmascarado del −1 · **sin `cxr_view`** · **calibración isotónica** en VAL · **OOF sin fuga** · nombres legibles.

> **Aviso:** con el nuevo etiquetado y la nueva métrica, los resultados **no son comparables** con la ejecución previa
> (que derivaba negativos de *No Finding* y seleccionaba por AUC-ROC). Hay que reentrenar.

In [ ]:
# CELDA 1 · DEPENDENCIAS
import subprocess, sys
for p in ["xgboost","scikit-learn","pandas","numpy","matplotlib","seaborn"]:
    subprocess.run([sys.executable,"-m","pip","install",p,"-q"], check=False)
print("Dependencias listas.")

In [ ]:
# CELDA 2 · IMPORTS, RUTAS Y CONSTANTES
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import KFold
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             confusion_matrix, roc_curve, precision_recall_curve)
import xgboost as xgb
warnings.filterwarnings("ignore"); SEED=42; np.random.seed(SEED)

BASE=Path(r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0")
CSV=BASE/"data_csv"/"clean"
TRAIN_CSV,VAL_CSV,TEST_CSV=CSV/"train_clean.csv",CSV/"val_clean.csv",CSV/"test_clean.csv"
OUTPUT_DIR=Path(r"C:\TFM\1.Opción - Symile Mimic\tfm_multimodal_clinico\salidas\03_labs\v1"); OUTPUT_DIR.mkdir(exist_ok=True); FIG=OUTPUT_DIR/"figuras"; FIG.mkdir(exist_ok=True)

LABELS=["Atelectasis","Cardiomegaly","Edema","Lung Opacity","No Finding","Pleural Effusion"]
N_LABELS=len(LABELS); NO_FINDING="No Finding"
PATHOLOGY=[l for l in LABELS if l!=NO_FINDING]; CORE=["Cardiomegaly","Edema","Pleural Effusion"]
K_FOLDS=5; FLAG_THRESH=0.02
GENDER={0:0,1:1,"0":0,"1":1,"M":1,"F":0}; RACE={"UNKNOWN":0,"WHITE":1,"BLACK":2,"ASIAN":3,"HISPANIC_LATINO":4,"OTHER_KNOWN":0}
ADM={"SCHEDULED":0,"EMERGENCY":1,"OBSERVATION":2,"URGENT":3}; LOC={"EMERGENCY_ROOM":0,"REFERRAL":1,"TRANSFER":2,"INTRA_HOSPITAL":3}
print("Salidas en", OUTPUT_DIR)

In [ ]:
# CELDA 3 · CARGA + OBJETIVOS (definición FINAL del negativo)
df_train=pd.read_csv(TRAIN_CSV,sep=";"); df_val=pd.read_csv(VAL_CSV,sep=";"); df_test=pd.read_csv(TEST_CSV,sep=";")
DEMO=["subject_id","hadm_id","cxr_path","ecg_path","age","gender","race","admission_type","admission_location","cxr_view","hours_adm_to_cxr"]
RAW_LABS=[c for c in df_train.columns if c not in LABELS and c not in DEMO and "pctile" not in c]

# ══ build_targets ════════════════════════════════════════════════════════════
# QUÉ HACE: convierte los estados 1/0/−1/NaN de cada etiqueta en dos matrices
#           (labels, mask) para el entrenamiento multietiqueta enmascarado.
# FINALIDAD: fijar la definición FINAL del negativo acordada con el tutor:
#            POS = (==1) ; NEG = (==0) | (NaN→0) ; el −1 (incierto) se ENMASCARA
#            (fuera de pérdida y métrica, estrategia U-Ignore de CheXpert).
# ORIGEN EDA: §3 "Definición del negativo" · recuadro naranja (NaN→0, U-Ignore,
#            «No Finding» NO como negativo para evitar el sesgo de espectro).
# CAMBIO vs versión previa: se ELIMINA la derivación de negativos desde «No Finding»
#            (derive=False por defecto). El NaN pasa a negativo (mask=1), no a enmascarado.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#            comparar el AUC-PR con/sin esta definición para justificar el cambio;
#            verificar que pos_weight ≈ 2,28/1,85/3,43/2,15/6,31/1,66 (train).
def build_targets(df, uncertainty_policy="ignore", derive=False, verbose=False):
    raw=df[LABELS].to_numpy(dtype=float); N=raw.shape[0]
    labels=(raw==1.0).astype(np.float32)                 # 1 → positivo ; 0 y NaN → 0 (negativo)
    mask  =np.ones((N,N_LABELS),np.float32)              # por defecto TODO entra en pérdida/métrica
    unc=(raw==-1.0)
    if   uncertainty_policy=="ignore": mask[unc]=0.0      # −1 → ENMASCARADO (definición FINAL)
    elif uncertainty_policy=="ones":   labels[unc]=1.0    # −1 → positivo (alternativa, no usada)
    # "zeros": −1 → negativo (ya es 0 en labels; nada que hacer)
    ndp=ndn=0
    if derive:  # CONSERVADO por compatibilidad; NO forma parte de la definición FINAL (sesgo de espectro)
        nf=LABELS.index(NO_FINDING); pc=[j for j in range(N_LABELS) if j!=nf]; nfp=(raw[:,nf]==1)
        for j in pc:
            f=nfp&np.isnan(raw[:,j]); labels[f,j]=0.0; ndp+=int(f.sum())
        ap=(raw[:,pc]==1).any(1); fn=ap&np.isnan(raw[:,nf]); labels[fn,nf]=0.0; ndn=int(fn.sum())
    if verbose:
        obs=mask==1
        print(f"   pos={int((labels*mask).sum()):,} · neg={int(((labels==0)&obs).sum()):,} · enmascarados(−1)={int((mask==0).sum()):,}"
              + (f" · derivados(off)={ndp+ndn:,}" if derive else ""))
    return labels,mask
y_train,m_train=build_targets(df_train,verbose=True); y_val,m_val=build_targets(df_val); y_test,m_test=build_targets(df_test)
print(f"train={len(df_train)} val={len(df_val)} test={len(df_test)} · {len(RAW_LABS)} analíticas")

In [ ]:
# CELDA 4 · FEATURES: analíticas (NaN nativo) + ratios clínicos + flags de missingness + demografía
# DECISIONES DE FEATURES (derivadas del EDA), válidas para esta v1 (árboles):
#  · RAW vs PERCENTIL: se usa el valor RAW (los árboles son invariantes a la escala y manejan NaN
#    de forma nativa). Raw≡percentil (r>0,90, §9 EDA) → NO se meten ambos (evita duplicar dimensión).
#  · NaN AMPLIOS: ninguna analítica supera el 70 % (§4 EDA) → se conservan todas. El árbol usa el NaN
#    como información; la regla "quedarse solo con el flag" se reserva a los modelos DENSOS (v2/v3).
#  · CLUSTERS FISIOLÓGICOS: la reducción por representante NO se aplica en árboles (robustos a la
#    colinealidad). Se documenta para los modelos densos (v2/v3), donde sí puede ayudar a generalizar.
#  · PRIORITARIAS (Urea, Albúmina, RDW, Creatinina; §5/§10 EDA): presentes y protegidas (nunca se descartan).
train_miss=df_train[RAW_LABS].isna().mean(); FLAG_LABS=[c for c in RAW_LABS if train_miss[c]>FLAG_THRESH]
def _col(key):
    for c in RAW_LABS:
        if c.startswith(key+"_") or c==key: return c
    return None
C_UREA=_col("urea_nitrogen"); C_CREAT=_col("creatinine"); C_NEUT=_col("neutrophils_pct"); C_LYMPH=_col("lymphocytes_pct"); C_RDW=_col("rdw")

# ══ ratios ═══════════════════════════════════════════════════════════════════
# QUÉ HACE: construye ratios clínicos derivados del valor bruto.
# FINALIDAD: aportar señal fisiológica que una analítica sola no captura:
#            BUN/Creatinina (función renal), Neutrófilos/Linfocitos (inflamación), RDW×Edad (riesgo cardio).
# ORIGEN EDA: §5/§10 · recuadro naranja "priorizar Urea, Albúmina, RDW, Creatinina y añadir ratios clínicos".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): comprobar en importancia si los ratios superan a sus
#            componentes; si RDW×Edad domina, reforzar el eje cardio-renal en la discusión.
def ratios(df):
    g=lambda c: df[c].to_numpy(np.float32) if c else np.full(len(df),np.nan,np.float32)
    urea,creat,neut,lymph,rdw=g(C_UREA),g(C_CREAT),g(C_NEUT),g(C_LYMPH),g(C_RDW); age=df["age"].astype(float).to_numpy(np.float32)
    out=np.vstack([urea/(creat+1e-6), neut/(lymph+1e-6), rdw*age/100.0]).T.astype(np.float32)
    return out,["BUN/Creatinina","Neutrófilos/Linfocitos","RDW×Edad"]

# ══ demo ═════════════════════════════════════════════════════════════════════
# QUÉ HACE: matriz de variables demográficas (edad, sexo, raza/ingreso/lugar one-hot, horas→CXR).
# FINALIDAD: incluir la EDAD (predictor tabular más fuerte, §10 EDA) y el contexto de ingreso.
# ORIGEN EDA: §2 · recuadro naranja "«Desconocida» es categoría, no NaN → one-hot, no imputar";
#            §10 "la edad domina el ranking IVF (empatada con Linfocitos %)".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): evaluar equidad por sexo/raza (§9, V de Cramér ≤0,05).
def demo(df):
    n=len(df); cols=[df["age"].astype(float).to_numpy()[:,None], df["gender"].map(lambda v:float(GENDER.get(v,0))).to_numpy()[:,None]]
    names=["Edad","Sexo"]
    def oh(s,mp,pref):
        M=np.zeros((n,max(mp.values())+1),np.float32)
        for i,v in enumerate(s): M[i,mp.get(str(v).upper(),0)]=1.0
        return M,[f"{pref}={k}" for k,_ in sorted(mp.items(),key=lambda x:x[1])][:M.shape[1]]
    for c,mp,pref in [("race",RACE,"Raza"),("admission_type",ADM,"Ingreso"),("admission_location",LOC,"Lugar")]:
        M,nm=oh(df[c],mp,pref); cols.append(M); names+=nm
    cols.append(df["hours_adm_to_cxr"].astype(float).to_numpy()[:,None]); names.append("Horas ingreso→CXR")
    return np.hstack(cols).astype(np.float32),names

# ══ features ═════════════════════════════════════════════════════════════════
# QUÉ HACE: ensambla la matriz final = analíticas RAW + ratios + flags de missingness + demografía.
# FINALIDAD: alimentar al XGBoost con la señal tabular completa (sin imputar; NaN nativo).
# ORIGEN EDA: §6 · recuadro naranja "incluir flags de missingness (MNAR): los positivos tienen menos ausencias".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): vigilar que el modelo NO dependa en exceso de los flags
#            (atajo espurio "tiene analíticas ⇒ enfermo"); revisar su importancia.
def features(df):
    r,rn=ratios(df); d,dn=demo(df)
    X=np.hstack([df[RAW_LABS].to_numpy(np.float32), r, df[FLAG_LABS].isna().astype(np.float32).to_numpy(), d])
    names=[c.rsplit("_",1)[0].replace("_"," ").title() for c in RAW_LABS]+rn+[f"falta:{c.rsplit('_',1)[0]}" for c in FLAG_LABS]+dn
    return X,names
X_train,FEAT_NAMES=features(df_train); X_val,_=features(df_val); X_test,_=features(df_test)
print(f"Features: {X_train.shape[1]} ({len(FLAG_LABS)} flags de missingness) · XGBoost maneja los NaN nativamente")

In [ ]:
# CELDA 5 · MÉTRICAS + XGBoost one-vs-rest (con masking y balanceo por etiqueta)

# ══ multilabel_metrics ═══════════════════════════════════════════════════════
# QUÉ HACE: calcula por etiqueta AUC-ROC, AUC-PR (Average Precision), F1, sens/spec
#           y la confusión, respetando la máscara (los −1 no cuentan).
# FINALIDAD: evaluar el modelo con la MÉTRICA PRIMARIA = AUC-PR (macro_AP_path);
#            el AUC-ROC pasa a secundario.
# ORIGEN EDA: §3 · recuadro naranja "AUC-PR primaria + IC bootstrap; con desbalanceo
#            la AP es más honesta; NO comparar AP entre patologías de distinta prevalencia".
# INTERPRETACIÓN FUTURA (→ recuadro naranja doc): reportar AP por etiqueta con su IC;
#            recordar que la línea base de AP = prevalencia (no comparable entre etiquetas).
def multilabel_metrics(probs,labels,mask,thresholds=None):
    if thresholds is None: thresholds={l:0.5 for l in LABELS}
    res={}
    for j,l in enumerate(LABELS):
        s=mask[:,j]==1; yt=labels[s,j]; yp=probs[s,j]; npos=int(yt.sum()); nneg=int((1-yt).sum())
        pred=(yp>=thresholds.get(l,0.5)).astype(float)
        auc=roc_auc_score(yt,yp) if npos>=2 and nneg>=2 else float("nan")
        ap=average_precision_score(yt,yp) if npos>=2 and nneg>=2 else float("nan")
        tp=int(((pred==1)&(yt==1)).sum()); tn=int(((pred==0)&(yt==0)).sum())
        fp=int(((pred==1)&(yt==0)).sum()); fn=int(((pred==0)&(yt==1)).sum())
        res[l]={"AUC":auc,"AP":ap,"F1":f1_score(yt,pred,zero_division=0),"sens":tp/max(tp+fn,1),"spec":tn/max(tn+fp,1),
                "n_pos":npos,"n_neg":nneg,"TP":tp,"TN":tn,"FP":fp,"FN":fn,"thr":thresholds.get(l,0.5)}
    mac =lambda g,k:float(np.nanmean([res[l][k] for l in g])) if any(not np.isnan(res[l][k]) for l in g) else float("nan")
    res["macro_AUC_core"]=mac(CORE,"AUC"); res["macro_AUC_path"]=mac(PATHOLOGY,"AUC")   # secundaria (ROC)
    res["macro_AP_core"] =mac(CORE,"AP");  res["macro_AP_path"] =mac(PATHOLOGY,"AP")    # PRIMARIA (AUC-PR)
    return res

# ══ bootstrap_ap_ci ══════════════════════════════════════════════════════════
# QUÉ HACE: intervalo de confianza (percentil) del AUC-PR por remuestreo bootstrap.
# FINALIDAD: acompañar SIEMPRE la AP de su IC, porque val/test son pequeños (§1 EDA)
#            y la métrica tiene varianza alta.
# ORIGEN EDA: §1 · recuadro naranja "reportar IC bootstrap por el tamaño reducido de val/test".
def bootstrap_ap_ci(probs,labels,mask,n_boot=1000,alpha=0.05,seed=SEED):
    rng=np.random.RandomState(seed); out={}
    for j,l in enumerate(LABELS):
        s=np.where(mask[:,j]==1)[0]; yt=labels[s,j]; yp=probs[s,j]
        if int(yt.sum())<2 or int((1-yt).sum())<2: out[l]=(float("nan"),float("nan")); continue
        vals=[]
        for _ in range(n_boot):
            idx=rng.randint(0,len(s),len(s))
            if yt[idx].sum()<1 or (1-yt[idx]).sum()<1: continue
            vals.append(average_precision_score(yt[idx],yp[idx]))
        out[l]=(float(np.percentile(vals,100*alpha/2)),float(np.percentile(vals,100*(1-alpha/2)))) if vals else (float("nan"),float("nan"))
    return out

# ══ best_thresholds_by_f1 ════════════════════════════════════════════════════
# QUÉ HACE: busca el umbral por etiqueta que maximiza F1 sobre un conjunto (VAL).
# FINALIDAD: fijar puntos de operación en VALIDACIÓN (no en test), según el protocolo.
# ORIGEN EDA: §11 (decisiones) · "VAL calibra + umbrales; TEST una sola vez".
def best_thresholds_by_f1(probs,labels,mask):
    grid=np.linspace(0.05,0.95,37); thr={}
    for j,l in enumerate(LABELS):
        s=mask[:,j]==1; yt=labels[s,j]; yp=probs[s,j]
        if yt.sum()<2: thr[l]=0.5; continue
        bf,bt=-1,0.5
        for t in grid:
            f=f1_score(yt,(yp>=t).astype(float),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[l]=float(bt)
    return thr

# ══ make_xgb ═════════════════════════════════════════════════════════════════
# QUÉ HACE: crea un XGBoost binario con scale_pos_weight por etiqueta.
# FINALIDAD: atacar el desbalanceo moderado (1,7:1–6,3:1) con scale_pos_weight = n_neg/n_pos;
#            los árboles manejan el NaN de forma nativa (no se imputa) y usan el valor RAW.
# ORIGEN EDA: §3 (desbalanceo → scale_pos_weight, nunca SMOTE) · §4 (NaN nativo, raw en árboles).
def make_xgb(spw):
    return xgb.XGBClassifier(n_estimators=600,max_depth=4,learning_rate=0.03,subsample=0.8,colsample_bytree=0.8,
                             min_child_weight=2,reg_lambda=1.5,tree_method="hist",eval_metric="logloss",
                             n_jobs=4,random_state=SEED,scale_pos_weight=spw)

# ══ fit_predict_ovr ══════════════════════════════════════════════════════════
# QUÉ HACE: entrena un clasificador binario por etiqueta (one-vs-rest) y predice.
# FINALIDAD: tratar las 6 salidas como binary relevance INDEPENDIENTES.
# ORIGEN EDA: §3 · recuadro naranja "Jaccard ≤ 0,30 → BCE/OvR independiente por etiqueta
#            (no hace falta modelar dependencias entre salidas)".
def fit_predict_ovr(Xtr,ytr,mtr,Xva):
    P=np.full((len(Xva),N_LABELS),0.5,np.float32)
    for j in range(N_LABELS):
        sel=mtr[:,j]==1; Xj=Xtr[sel]; yj=ytr[sel,j]
        if len(np.unique(yj))<2: P[:,j]=float(yj.mean()) if len(yj) else 0.5; continue
        spw=(yj==0).sum()/max((yj==1).sum(),1); clf=make_xgb(spw); clf.fit(Xj,yj); P[:,j]=clf.predict_proba(Xva)[:,1]
    return P
print("Métricas (AUC-PR primaria + IC bootstrap) y XGBoost listos.")

In [ ]:
# CELDA 5b · KIT DE EVALUACIÓN CLÍNICA (B1–B8 del plan) — derivado de los recuadros naranjas del EDA
# Este bloque implementa lo que el EDA pedía y no estaba: puntos de operación clínicos, verificación
# de la calibración, IC bootstrap ampliado, importancia MI/ANOVA, chequeo de consistencia,
# dependencia de los flags MNAR y estratificación por subgrupos.
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.feature_selection import mutual_info_classif, f_classif

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · bootstrap_ci_metric                                          [B4]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : remuestrea con reemplazo las observaciones válidas (mask==1) y recalcula la métrica
#              indicada, devolviendo el intervalo percentil. Lo hace POR ETIQUETA y para el MACRO.
# POR QUÉ    : una métrica puntual sobre 464 pacientes no dice si una diferencia entre modelos es
#              real o azar; el IC es lo que permite afirmar (o no) que un modelo supera a otro.
# ENTRADAS   : probs (N,6) probabilidades · labels (N,6) 0/1 · mask (N,6) 1=cuenta
#              metric: "ap" (AUC-PR) o "auc" (AUC-ROC) · n_boot · alpha (0.05 → IC95 %)
# SALIDAS    : dict {etiqueta: (lo, hi)} + clave "macro_path" con el IC del macro de las 5 patologías
# ORIGEN EDA : §1 · recuadro naranja "reportar SIEMPRE IC bootstrap en las métricas por el tamaño
#              reducido de val/test (750 y 464)".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              si los IC de dos modelos (o de fusión vs mejor mono-modelo) SE SOLAPAN, hay que decir
#              explícitamente que la mejora no es concluyente. Es un punto que el tribunal valorará.
# ══════════════════════════════════════════════════════════════════════════════
def bootstrap_ci_metric(probs, labels, mask, metric="ap", n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.RandomState(seed)
    scorer = average_precision_score if metric == "ap" else roc_auc_score
    out, per_boot_macro = {}, []
    for j, l in enumerate(LABELS):
        idx_valid = np.where(mask[:, j] == 1)[0]
        yt, yp = labels[idx_valid, j], probs[idx_valid, j]
        if int(yt.sum()) < 2 or int((1 - yt).sum()) < 2:
            out[l] = (float("nan"), float("nan")); continue
        vals = []
        for b in range(n_boot):
            bs = rng.randint(0, len(idx_valid), len(idx_valid))
            if yt[bs].sum() < 1 or (1 - yt[bs]).sum() < 1: continue
            vals.append(scorer(yt[bs], yp[bs]))
        out[l] = (float(np.percentile(vals, 100*alpha/2)), float(np.percentile(vals, 100*(1-alpha/2)))) if vals else (float("nan"),)*2
    # IC del MACRO: se remuestrean PACIENTES (no etiquetas) para respetar la correlación entre salidas
    for b in range(n_boot):
        bs = rng.randint(0, len(probs), len(probs)); per_label = []
        for j, l in enumerate(LABELS):
            if l not in PATHOLOGY: continue
            sel = mask[bs, j] == 1; yt, yp = labels[bs][sel, j], probs[bs][sel, j]
            if yt.sum() < 1 or (1 - yt).sum() < 1: continue
            per_label.append(scorer(yt, yp))
        if per_label: per_boot_macro.append(np.mean(per_label))
    out["macro_path"] = (float(np.percentile(per_boot_macro, 100*alpha/2)),
                         float(np.percentile(per_boot_macro, 100*(1-alpha/2)))) if per_boot_macro else (float("nan"),)*2
    return out

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · operating_points                                             [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : para cada etiqueta busca en VALIDACIÓN tres umbrales de decisión:
#                1) "f1"       → máximo F1 (equilibrio, uso general)
#                2) "cribado"  → el umbral MÁS ALTO que aún garantiza Sensibilidad ≥ sens_target
#                3) "confirm"  → el umbral MÁS BAJO que aún garantiza Especificidad ≥ spec_target
# POR QUÉ    : un único umbral no sirve en clínica. Para CRIBAR interesa no dejar escapar enfermos
#              (alta sensibilidad, se aceptan falsas alarmas); para CONFIRMAR interesa no alarmar en
#              falso (alta especificidad). Son dos decisiones clínicas distintas sobre el mismo modelo.
# ENTRADAS   : probs/labels/mask del conjunto de VALIDACIÓN · sens_target · spec_target
# SALIDAS    : dict {modo: {etiqueta: umbral}} con modos "f1", "cribado" y "confirm"
# ORIGEN EDA : §11 (tabla de decisiones) · "Puntos de operación: fijar en VAL alta sensibilidad
#              (cribado) y alta especificidad (confirmación). Reportar Se/Sp/VPP/VPN".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              el VPP depende de la PREVALENCIA: con prevalencias del 13–36 % un VPP modesto puede ser
#              aceptable para cribado pero inservible para confirmar. Hay que discutir cada punto de
#              operación en términos de la consecuencia clínica del error, no solo del número.
# ══════════════════════════════════════════════════════════════════════════════
def operating_points(probs, labels, mask, sens_target=0.90, spec_target=0.90):
    grid = np.linspace(0.01, 0.99, 99)
    pts = {"f1": {}, "cribado": {}, "confirm": {}}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if yt.sum() < 2 or (1 - yt).sum() < 2:
            for k in pts: pts[k][l] = 0.5
            continue
        best_f1, thr_f1 = -1, 0.5; thr_sens, thr_spec = grid[0], grid[-1]
        for t in grid:
            pred = (yp >= t).astype(float)
            tp = ((pred == 1) & (yt == 1)).sum(); fn = ((pred == 0) & (yt == 1)).sum()
            tn = ((pred == 0) & (yt == 0)).sum(); fp = ((pred == 1) & (yt == 0)).sum()
            sens = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
            f1 = f1_score(yt, pred, zero_division=0)
            if f1 > best_f1: best_f1, thr_f1 = f1, t
            if sens >= sens_target: thr_sens = max(thr_sens, t)   # el más exigente que aún criba bien
            if spec >= spec_target: thr_spec = min(thr_spec, t)   # el más permisivo que aún confirma bien
        pts["f1"][l], pts["cribado"][l], pts["confirm"][l] = float(thr_f1), float(thr_sens), float(thr_spec)
    return pts

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · clinical_report                                              [B1]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : evalúa un conjunto de umbrales y devuelve, por etiqueta, las métricas que un clínico
#              interpreta directamente: Sensibilidad, Especificidad, VPP, VPN y la confusión.
# POR QUÉ    : AUC y AP resumen el ranking, pero la decisión real se toma en UN umbral; el clínico
#              necesita saber a cuántos enfermos se le escapan y cuántas alarmas falsas genera.
# ENTRADAS   : probs/labels/mask (TEST) · thresholds {etiqueta: umbral} · nombre del punto de operación
# SALIDAS    : DataFrame con una fila por etiqueta (punto, umbral, Se, Sp, VPP, VPN, TP/TN/FP/FN, prev)
# ORIGEN EDA : §11 · "Reportar Se/Sp/VPP/VPN" · §3 (la prevalencia condiciona el VPP).
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar el nº de FALSOS NEGATIVOS en el punto de cribado (lo que de verdad importa
#              para no perder un derrame) frente a los FALSOS POSITIVOS en el punto de confirmación.
# ══════════════════════════════════════════════════════════════════════════════
def clinical_report(probs, labels, mask, thresholds, punto="f1"):
    rows = []
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        t = thresholds.get(l, 0.5); pred = (yp >= t).astype(float)
        tp = int(((pred == 1) & (yt == 1)).sum()); tn = int(((pred == 0) & (yt == 0)).sum())
        fp = int(((pred == 1) & (yt == 0)).sum()); fn = int(((pred == 0) & (yt == 1)).sum())
        rows.append({"punto": punto, "etiqueta": l, "umbral": round(t, 3),
                     "Se": tp/max(tp+fn,1), "Sp": tn/max(tn+fp,1),
                     "VPP": tp/max(tp+fp,1), "VPN": tn/max(tn+fn,1),
                     "TP": tp, "TN": tn, "FP": fp, "FN": fn,
                     "prevalencia": (tp+fn)/max(tp+tn+fp+fn,1)})
    return pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · calibration_report                                           [B2]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : calcula el Brier score por etiqueta y los puntos de la curva de fiabilidad
#              (probabilidad predicha media vs frecuencia observada, en 10 bins).
# POR QUÉ    : la herramienta clínica muestra PROBABILIDADES; si no están calibradas, un 0,8 no
#              significa "80 % de estos pacientes lo tienen" y la cifra engaña al médico.
#              El Brier mide el error cuadrático medio de la probabilidad (menor = mejor).
# ENTRADAS   : probs/labels/mask · n_bins
# SALIDAS    : (DataFrame Brier por etiqueta, dict {etiqueta: (frac_positivos, media_predicha)})
# ORIGEN EDA : §11 · "Calibración: Platt/isotónica en VAL; verificar con Brier score y curva de
#              fiabilidad; recomprobar tras fusión".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              comparar Brier ANTES vs DESPUÉS de la isotónica. Si no mejora, la calibración no está
#              aportando y hay que decirlo. OJO: hay que RECALIBRAR tras la fusión multimodal.
# ══════════════════════════════════════════════════════════════════════════════
def calibration_report(probs, labels, mask, n_bins=10):
    rows, curves = [], {}
    for j, l in enumerate(LABELS):
        sel = mask[:, j] == 1; yt, yp = labels[sel, j], probs[sel, j]
        if len(np.unique(yt)) < 2:
            rows.append({"etiqueta": l, "Brier": float("nan")}); continue
        rows.append({"etiqueta": l, "Brier": float(brier_score_loss(yt, np.clip(yp, 0, 1)))})
        try:
            frac, mean_pred = calibration_curve(yt, np.clip(yp, 0, 1), n_bins=n_bins, strategy="quantile")
            curves[l] = (frac, mean_pred)
        except Exception:
            curves[l] = (np.array([]), np.array([]))
    return pd.DataFrame(rows), curves

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · consistency_no_finding                                       [B6]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : mide la coherencia entre P(«Sin hallazgo») y la probabilidad máxima de patología.
#              Cuenta como INCOHERENTE el caso en que ambas superan 0,5 a la vez (el modelo afirma
#              simultáneamente "está sano" y "tiene una patología").
# POR QUÉ    : las 6 cabezas son independientes (binary relevance), así que nada las obliga a ser
#              coherentes entre sí. Este chequeo detecta si el modelo se contradice, algo inaceptable
#              en una herramienta que un clínico va a leer.
# ENTRADAS   : probs (N,6)
# SALIDAS    : dict con la correlación de Pearson (debería ser NEGATIVA) y el % de casos incoherentes
# ORIGEN EDA : §3 · recuadro naranja "«Sin hallazgo» se modela como una etiqueta más (P(normal) útil
#              como chequeo de consistencia), nunca como fuente de negativos".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              una correlación cercana a 0 o positiva indicaría que la cabeza «Sin hallazgo» no está
#              aprendiendo el concepto de normalidad → argumento para revisarla o para no mostrarla
#              en la herramienta clínica.
# ══════════════════════════════════════════════════════════════════════════════
def consistency_no_finding(probs):
    j_nf = LABELS.index(NO_FINDING); j_path = [j for j in range(N_LABELS) if j != j_nf]
    p_nf = probs[:, j_nf]; p_max = probs[:, j_path].max(axis=1)
    incoh = float(((p_nf > 0.5) & (p_max > 0.5)).mean() * 100)
    corr = float(np.corrcoef(p_nf, p_max)[0, 1])
    return {"corr_NoFinding_vs_maxPatologia": corr, "pct_incoherentes": incoh}

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · flag_importance_share                                        [B7]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : suma la importancia que el modelo asigna a los flags de missingness y la expresa
#              como % del total.
# POR QUÉ    : el EDA demostró que la ausencia de analíticas es MNAR (a los graves se les piden más
#              pruebas). Eso es señal útil, pero si el modelo se apoya demasiado en ella estará
#              aprendiendo "le hicieron analítica ⇒ está enfermo" en vez de la biología.
# ENTRADAS   : imp (vector de importancias) · names (nombres de features, los flags empiezan por "falta:")
# SALIDAS    : dict con el % de importancia en flags y el nombre del flag más influyente
# ORIGEN EDA : §6 · recuadro naranja "vigilar que el modelo no dependa en exceso del patrón de
#              ausencia (revisar importancia de los flags)".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              si el % supera ~25-30 %, advertir en la memoria de que parte del rendimiento es un
#              artefacto del proceso asistencial y NO generalizaría a otro hospital con otra política
#              de peticiones analíticas.
# ══════════════════════════════════════════════════════════════════════════════
def flag_importance_share(imp, names):
    imp = np.asarray(imp, dtype=float); tot = imp.sum() + 1e-12
    idx = [i for i, n in enumerate(names) if str(n).startswith("falta:")]
    if not idx: return {"pct_importancia_flags": 0.0, "flag_top": None}
    share = float(imp[idx].sum() / tot * 100)
    top = names[idx[int(np.argmax(imp[idx]))]]
    return {"pct_importancia_flags": share, "flag_top": top}

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · stratified_report                                            [B8]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : recalcula la métrica primaria (AUC-PR macro) dentro de cada subgrupo demográfico y,
#              además, por PROYECCIÓN radiográfica (cxr_view).
# POR QUÉ    : (a) equidad — comprobar que el modelo no rinde peor en un grupo;
#              (b) robustez — la placa AP se hace al paciente encamado (más grave) y magnifica la
#              silueta cardíaca, así que conviene ver si el rendimiento depende de la proyección.
# ENTRADAS   : df (metadatos del split evaluado) · probs/labels/mask · cols (columnas a estratificar)
# SALIDAS    : DataFrame (variable, grupo, n, macro_AP, macro_AUC)
# ORIGEN EDA : §2/§9 · "diferencias demográficas pequeñas pero reales → evaluar equidad por sexo y
#              etnia" y "monitorizar equidad". ANEXO CXR · "considerar cxr_view como covariable o al
#              menos monitorizar su efecto en Cardiomegalia".
# DECISIÓN DE DISEÑO: cxr_view se usa SOLO aquí, para estratificar. NO entra como variable predictora
#              en ningún modelo: al ser un proxy de "paciente encamado/grave", incluirla inflaría el
#              resultado mediante un atajo asistencial en vez de señal biológica.
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              subgrupos con n<60 (varias etnias en el test de 464) pueden dar diferencias por PURO
#              RUIDO; no afirmar inequidad sin intervalo de confianza. Si el rendimiento cae mucho en
#              PA frente a AP, discutir el sesgo de espectro de la cohorte.
# ══════════════════════════════════════════════════════════════════════════════
def stratified_report(df, probs, labels, mask, cols=("gender", "race", "admission_type", "cxr_view")):
    d = df.reset_index(drop=True); rows = []
    for col in cols:
        if col not in d.columns: continue
        for v in sorted(d[col].dropna().unique(), key=str):
            idx = d.index[d[col] == v].to_numpy()
            if len(idx) < 15:
                rows.append({"variable": col, "grupo": str(v), "n": len(idx),
                             "macro_AP": np.nan, "macro_AUC": np.nan}); continue
            mm = multilabel_metrics(probs[idx], labels[idx], mask[idx])
            rows.append({"variable": col, "grupo": str(v), "n": len(idx),
                         "macro_AP": mm["macro_AP_path"], "macro_AUC": mm["macro_AUC_path"]})
    return pd.DataFrame(rows)

# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · importancia_mi_anova                                         [B5]
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : calcula la importancia de cada variable por Información Mutua y por F de ANOVA,
#              promediando sobre las etiquetas (normalizando cada una a máximo 1).
# POR QUÉ    : la importancia por Gini de los árboles se SESGA con clases desbalanceadas y con
#              variables de muchos cortes posibles; MI y ANOVA son criterios independientes que
#              permiten comprobar si el ranking se sostiene.
# ENTRADAS   : X (N,F) matriz de features (se imputa la mediana) · y (N,6) · m (N,6) · names
# SALIDAS    : DataFrame ordenado con importancia_MI e importancia_ANOVA normalizadas
# ORIGEN EDA : §10 · recuadro naranja "en Opacidad, fiarse más de MI/ANOVA que de RF-Gini por el
#              desbalanceo" y "los tres métodos coinciden → reduce el riesgo de artefacto".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              contrastar este ranking con el de Gini. Donde coincidan, la conclusión es sólida;
#              donde discrepen (esperable en Lung Opacity), primar MI/ANOVA y decirlo.
# ══════════════════════════════════════════════════════════════════════════════
def importancia_mi_anova(X, y, m, names, seed=SEED):
    Xi = np.where(np.isnan(X), np.nanmedian(np.where(np.isnan(X), np.nan, X), axis=0), X)
    Xi = np.nan_to_num(Xi, nan=0.0, posinf=0.0, neginf=0.0)
    acc_mi = np.zeros(Xi.shape[1]); acc_f = np.zeros(Xi.shape[1]); n_ok = 0
    for j in range(N_LABELS):
        sel = m[:, j] == 1; yj = y[sel, j]
        if len(np.unique(yj)) < 2: continue
        mi = mutual_info_classif(Xi[sel], yj, random_state=seed)
        fv = np.nan_to_num(f_classif(Xi[sel], yj)[0], nan=0.0, posinf=0.0)
        acc_mi += mi / (mi.max() + 1e-12); acc_f += fv / (fv.max() + 1e-12); n_ok += 1
    acc_mi /= max(n_ok, 1); acc_f /= max(n_ok, 1)
    return (pd.DataFrame({"variable": names, "importancia_MI": np.round(acc_mi, 4),
                          "importancia_ANOVA": np.round(acc_f, 4)})
            .assign(media=lambda d: (d.importancia_MI + d.importancia_ANOVA) / 2)
            .sort_values("media", ascending=False).reset_index(drop=True))

print("Kit de evaluación clínica listo: B1 puntos de operación · B2 calibración · B4 IC bootstrap ·")
print("                                B5 MI/ANOVA · B6 consistencia · B7 flags MNAR · B8 estratificación")

In [ ]:
# CELDA 6 · K-FOLD -> OOF de train (sin fuga) + predicción de val/test
# Métrica reportada = AUC-PR macro (primaria); se conserva AUC-ROC como secundaria.
kf=KFold(K_FOLDS,shuffle=True,random_state=SEED)
oof_train=np.zeros((len(df_train),N_LABELS),np.float32); fold_ap=[]; fold_auc=[]
for k,(tr,va) in enumerate(kf.split(np.arange(len(df_train)))):
    oof_train[va]=fit_predict_ovr(X_train[tr],y_train[tr],m_train[tr],X_train[va])
    mm=multilabel_metrics(oof_train[va],y_train[va],m_train[va]); fold_ap.append(mm["macro_AP_path"]); fold_auc.append(mm["macro_AUC_path"])
    print(f"Fold {k+1}/{K_FOLDS}: macroAP_path={mm['macro_AP_path']:.4f} · macroAUC_path={mm['macro_AUC_path']:.4f}")
val_pred_raw=fit_predict_ovr(X_train,y_train,m_train,X_val)
test_pred_raw=fit_predict_ovr(X_train,y_train,m_train,X_test)
print(f"\nOOF macroAP_path={np.nanmean(fold_ap):.4f}±{np.nanstd(fold_ap):.4f} (primaria) · macroAUC_path={np.nanmean(fold_auc):.4f} (secundaria)")

In [ ]:
# CELDA 7 · CALIBRACIÓN ISOTÓNICA (VAL) + VERIFICACIÓN (Brier) + PUNTOS DE OPERACIÓN + PRE-REGISTRO
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN · apply_cal
# ──────────────────────────────────────────────────────────────────────────────
# QUÉ HACE   : aplica a cada columna de probabilidades el regresor isotónico aprendido en VALIDACIÓN.
# POR QUÉ    : un XGBoost con scale_pos_weight produce puntuaciones ordenadas pero NO probabilidades
#              fiables; la isotónica las reajusta para que un 0,8 signifique de verdad ~80 %.
# ENTRADAS   : P (N,6) probabilidades crudas
# SALIDAS    : (N,6) probabilidades calibradas
# ORIGEN EDA : §11 · "Calibración: Platt/isotónica en VAL; verificar con Brier score y curva de
#              fiabilidad; recomprobar tras fusión".
# INTERPRETACIÓN FUTURA (→ recuadro naranja de la doc post-resultados):
#              la calibración se ajusta en VAL (750 pacientes); con tan pocos casos la isotónica puede
#              sobreajustar. Si el Brier NO mejora en test, decirlo y considerar Platt (más suave).
# ══════════════════════════════════════════════════════════════════════════════
calibrators = {}
for j, l in enumerate(LABELS):
    s = m_val[:, j] == 1; yt = y_val[s, j]; yp = val_pred_raw[s, j]
    calibrators[l] = None if len(np.unique(yt)) < 2 else IsotonicRegression(out_of_bounds="clip").fit(yp, yt)
def apply_cal(P):
    O = P.copy()
    for j, l in enumerate(LABELS):
        if calibrators[l] is not None: O[:, j] = calibrators[l].predict(P[:, j])
    return O
oof_cal = apply_cal(oof_train); val_pred = apply_cal(val_pred_raw); test_pred = apply_cal(test_pred_raw)

# ── B2 · ¿La calibración mejora realmente? Brier ANTES vs DESPUÉS (medido en VAL) ──────────────
brier_pre,  _       = calibration_report(val_pred_raw, y_val, m_val)
brier_post, curvas  = calibration_report(val_pred,     y_val, m_val)
cal_cmp = brier_pre.merge(brier_post, on="etiqueta", suffixes=("_sin_calibrar", "_calibrado"))
cal_cmp["mejora"] = cal_cmp["Brier_sin_calibrar"] - cal_cmp["Brier_calibrado"]
print("── B2 · Brier score en VALIDACIÓN (menor = mejor) ──")
print(cal_cmp.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"   Etiquetas en las que la calibración MEJORA el Brier: {(cal_cmp['mejora'] > 0).sum()}/{len(cal_cmp)}")
cal_cmp.to_csv(OUTPUT_DIR/"calibracion_brier.csv", index=False)

# Curva de fiabilidad (diagonal = calibración perfecta)
fig, ax = plt.subplots(figsize=(6.4, 6))
ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="calibración perfecta")
for l in LABELS:
    frac, mean_pred = curvas.get(l, (np.array([]), np.array([])))
    if len(frac): ax.plot(mean_pred, frac, "o-", ms=4, lw=1.6, label=l)
ax.set_xlabel("probabilidad predicha media"); ax.set_ylabel("frecuencia observada de positivos")
ax.set_title("Curva de fiabilidad tras calibración isotónica (VAL)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG/"curva_fiabilidad_v1.png", dpi=150, bbox_inches="tight"); plt.show()

# ── B1 · Tres puntos de operación fijados en VALIDACIÓN ────────────────────────────────────────
PUNTOS = operating_points(val_pred, y_val, m_val, sens_target=0.90, spec_target=0.90)
thr_val = PUNTOS["f1"]          # umbral por defecto (compatibilidad con el resto del notebook)
print("\n── B1 · Umbrales por punto de operación (fijados en VAL) ──")
print(f"{'Etiqueta':18s} {'F1':>6} {'Cribado(Se>=.90)':>18} {'Confirm(Sp>=.90)':>18}")
for l in LABELS:
    print(f"{l:18s} {PUNTOS['f1'][l]:6.2f} {PUNTOS['cribado'][l]:18.2f} {PUNTOS['confirm'][l]:18.2f}")

# ── B3 · PRE-REGISTRO de la regla de decisión (ANTES de tocar TEST) ────────────────────────────
# ORIGEN EDA: §11 · "Protocolo: TRAIN entrena; VAL calibra+umbrales+selección; TEST una sola vez.
#             Pre-registrar la regla de decisión."  Congelar aquí impide ajustar nada 'a posteriori'.
prereg = {
    "modelo": "LABS v1 · un XGBoost one-vs-rest",
    "etiquetado": "POS=(==1); NEG=(==0)|(NaN->0); -1 ENMASCARADO (U-Ignore); sin derivar de No Finding",
    "metrica_primaria": "AUC-PR (macro_AP_path); AUC-ROC secundaria; IC bootstrap 1000 remuestreos",
    "calibracion": "isotonica ajustada en VAL",
    "puntos_operacion": {"f1": PUNTOS["f1"], "cribado_Se>=0.90": PUNTOS["cribado"], "confirmacion_Sp>=0.90": PUNTOS["confirm"]},
    "cxr_view": "EXCLUIDA como predictor (proxy de gravedad); usada SOLO para estratificar resultados",
    "nota": "TEST se evalua UNA sola vez con estos umbrales; no se reajusta nada despues.",
}
json.dump(prereg, open(OUTPUT_DIR/"preregistro_regla_decision.json", "w", encoding="utf-8"),
          indent=2, ensure_ascii=False)
print("\n── B3 · Regla de decisión PRE-REGISTRADA en preregistro_regla_decision.json (test aún sin tocar)")

In [ ]:
# CELDA 8 · EVALUACIÓN EN TEST (UNA sola vez, con la regla PRE-REGISTRADA)
# Contenido: métricas de discriminación con IC (B4) · los 3 puntos de operación con Se/Sp/VPP/VPN (B1)
#            · consistencia de «Sin hallazgo» (B6) · estratificación por subgrupos y proyección (B8).
# PROTOCOLO (§11 EDA): calibración y umbrales vienen de VAL; aquí NO se ajusta nada.

# ── Discriminación por etiqueta, con IC bootstrap de AP y de AUC (B4) ─────────────────────────
M      = multilabel_metrics(test_pred, y_test, m_test, thresholds=thr_val)
ci_ap  = bootstrap_ci_metric(test_pred, y_test, m_test, metric="ap")
ci_auc = bootstrap_ci_metric(test_pred, y_test, m_test, metric="auc")
rows = []
print("── DISCRIMINACIÓN (test) · AP es la métrica primaria; su línea base es la PREVALENCIA ──")
print(f"{'Etiqueta':18s} {'AP':>7} {'IC95% AP':>16} {'prev':>6} {'AUC':>7} {'IC95% AUC':>16} {'N+':>5} {'N-':>5}")
print("-"*96)
for l in LABELS:
    m = M[l]; auc = f"{m['AUC']:.4f}" if not np.isnan(m['AUC']) else "  N/A"
    lo_a, hi_a = ci_ap[l]; lo_u, hi_u = ci_auc[l]
    f_ap  = f"[{lo_a:.3f},{hi_a:.3f}]" if not np.isnan(lo_a) else "        N/A"
    f_auc = f"[{lo_u:.3f},{hi_u:.3f}]" if not np.isnan(lo_u) else "        N/A"
    prev = m['n_pos']/max(m['n_pos']+m['n_neg'], 1)
    print(f"{l:18s} {m['AP']:7.4f} {f_ap:>16} {prev:6.3f} {auc:>7} {f_auc:>16} {m['n_pos']:5d} {m['n_neg']:5d}")
    rows.append({"label": l, **{k: m[k] for k in ['AUC','AP','F1','sens','spec','n_pos','n_neg']},
                 "prevalencia": prev, "AP_ci_lo": lo_a, "AP_ci_hi": hi_a, "AUC_ci_lo": lo_u, "AUC_ci_hi": hi_u,
                 "AP_supera_prevalencia": bool(m['AP'] > prev)})
print("-"*96)
mlo, mhi = ci_ap["macro_path"]; ulo, uhi = ci_auc["macro_path"]
print(f"MACRO patol.: AP={M['macro_AP_path']:.4f} IC95%=[{mlo:.3f},{mhi:.3f}]  (PRIMARIA)")
print(f"              AUC={M['macro_AUC_path']:.4f} IC95%=[{ulo:.3f},{uhi:.3f}]  (secundaria)")
n_ok = sum(r["AP_supera_prevalencia"] for r in rows if r["label"] in PATHOLOGY)
print(f"Patologías cuya AP supera su prevalencia (=hay señal real): {n_ok}/{len(PATHOLOGY)}")
pd.DataFrame(rows).to_csv(OUTPUT_DIR/"metrics_per_label_v1.csv", index=False)

# ── B1 · Los tres puntos de operación con métricas clínicas ───────────────────────────────────
print("\n── PUNTOS DE OPERACIÓN (test) · Se=sensibilidad, Sp=especificidad, VPP/VPN=valores predictivos ──")
clin = pd.concat([clinical_report(test_pred, y_test, m_test, PUNTOS[k], punto=nm)
                  for k, nm in [("f1","f1"), ("cribado","cribado_Se>=0.90"), ("confirm","confirmacion_Sp>=0.90")]],
                 ignore_index=True)
for punto in clin["punto"].unique():
    sub = clin[clin["punto"] == punto]
    print(f"\n  · Punto '{punto}':")
    print(f"    {'Etiqueta':18s} {'thr':>5} {'Se':>6} {'Sp':>6} {'VPP':>6} {'VPN':>6} {'FN':>4} {'FP':>4}")
    for _, r in sub.iterrows():
        print(f"    {r['etiqueta']:18s} {r['umbral']:5.2f} {r['Se']:6.3f} {r['Sp']:6.3f} {r['VPP']:6.3f} {r['VPN']:6.3f} {int(r['FN']):4d} {int(r['FP']):4d}")
clin.to_csv(OUTPUT_DIR/"puntos_operacion_test.csv", index=False)

# ── B6 · Coherencia interna de «Sin hallazgo» ─────────────────────────────────────────────────
cons = consistency_no_finding(test_pred)
print(f"\n── B6 · Consistencia: corr(P(Sin hallazgo), max P(patología))={cons['corr_NoFinding_vs_maxPatologia']:.3f} "
      f"(debe ser NEGATIVA) · casos incoherentes={cons['pct_incoherentes']:.1f} %")

# ── B8 · Equidad y robustez por subgrupo (incluye la proyección radiográfica) ─────────────────
strat = stratified_report(df_test, test_pred, y_test, m_test)
print("\n── B8 · Rendimiento por subgrupo (OJO: n<60 ⇒ posible ruido, no inequidad) ──")
print(strat.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
strat.to_csv(OUTPUT_DIR/"estratificacion_subgrupos.csv", index=False)

# ── Resumen máquina ───────────────────────────────────────────────────────────────────────────
json.dump({"version": "LABS v1 ligero (un XGBoost OvR)", "primary_metric": "AUC-PR (macro_AP_path)",
           "cv_macro_ap_path": float(np.nanmean(fold_ap)), "cv_macro_auc_path": float(np.nanmean(fold_auc)),
           "test_macro_ap_path": M["macro_AP_path"], "test_macro_ap_ci": [mlo, mhi],
           "test_macro_auc_path": M["macro_AUC_path"], "test_macro_auc_ci": [ulo, uhi],
           "test_per_label": {l: M[l] for l in LABELS},
           "consistencia_no_finding": cons,
           "preregistro": "preregistro_regla_decision.json"},
          open(OUTPUT_DIR/"summary_v1.json", "w", encoding="utf-8"), indent=2, default=str, ensure_ascii=False)
print("\nGuardados: metrics_per_label_v1.csv · puntos_operacion_test.csv · estratificacion_subgrupos.csv · summary_v1.json")

In [ ]:
# CELDA 9 · IMPORTANCIA DE VARIABLES: Gini vs MI vs ANOVA (B5) + dependencia de los flags MNAR (B7)
# ORIGEN EDA · §10: "en Opacidad, fiarse más de MI/ANOVA que de RF-Gini por el desbalanceo" y
#                   "los tres métodos coinciden → reduce el riesgo de artefacto de un método concreto".
#              §6: "vigilar que el modelo no dependa en exceso del patrón de ausencia (flags)".

# ── Importancia por Gini (la del propio XGBoost, promediada sobre las 6 etiquetas) ─────────────
imp = np.zeros(X_train.shape[1])
for j in range(N_LABELS):
    sel = m_train[:, j] == 1; yj = y_train[sel, j]
    if len(np.unique(yj)) < 2: continue
    spw = (yj == 0).sum()/max((yj == 1).sum(), 1); clf = make_xgb(spw); clf.fit(X_train[sel], yj)
    fi = clf.feature_importances_; imp += fi/(fi.sum() + 1e-9)
imp /= N_LABELS

# ── B5 · Importancia por Información Mutua y ANOVA (criterios independientes, no sesgados por Gini)
imp_alt = importancia_mi_anova(X_train, y_train, m_train, FEAT_NAMES)
imp_alt.to_csv(OUTPUT_DIR/"importancia_mi_anova.csv", index=False)
print("── B5 · Top-10 por Información Mutua + ANOVA (contrastar con Gini) ──")
print(imp_alt.head(10).to_string(index=False))

# ── B7 · ¿Cuánta de la importancia recae en los flags de missingness? ─────────────────────────
dep = flag_importance_share(imp, FEAT_NAMES)
print(f"\n── B7 · Importancia acumulada en flags MNAR: {dep['pct_importancia_flags']:.1f} % "
      f"(flag más influyente: {dep['flag_top']})")
if dep["pct_importancia_flags"] > 25:
    print("   [AVISO] Por encima del 25 %: parte del rendimiento puede ser un ARTEFACTO del proceso")
    print("           asistencial ('le pidieron analítica ⇒ está grave') y no generalizaría a otro hospital.")
json.dump(dep, open(OUTPUT_DIR/"dependencia_flags_mnar.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

# ── Figura: Gini (top 18) + comparación de los tres criterios ─────────────────────────────────
order = np.argsort(imp)[::-1][:18]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cols = ["#c0392b" if str(FEAT_NAMES[i]).startswith("falta:") else "#117a65" for i in order][::-1]
axes[0].barh([FEAT_NAMES[i] for i in order][::-1], [imp[i] for i in order][::-1], color=cols)
axes[0].set_title("Importancia Gini (XGBoost, top 18)\nrojo = flag de missingness")
top_alt = imp_alt.head(18).iloc[::-1]
axes[1].barh(top_alt["variable"], top_alt["media"], color="#5b8db8")
axes[1].set_title("Importancia media MI+ANOVA (top 18)")
plt.tight_layout(); plt.savefig(FIG/"importancia_v1.png", dpi=150, bbox_inches="tight"); plt.show()

# ── Figura: AP por etiqueta frente a su prevalencia (línea base real) ─────────────────────────
fig2, ax = plt.subplots(figsize=(9, 4.5))
aps   = [M[l]["AP"] for l in LABELS]
prevs = [M[l]["n_pos"]/max(M[l]["n_pos"]+M[l]["n_neg"], 1) for l in LABELS]
x = np.arange(N_LABELS)
ax.bar(x-0.2, aps, 0.4, label="AUC-PR obtenida", color="#117a65")
ax.bar(x+0.2, prevs, 0.4, label="prevalencia (línea base)", color="#bdc3c7")
ax.set_xticks(x); ax.set_xticklabels([l[:11] for l in LABELS], rotation=30, ha="right")
ax.set_ylim(0, 1); ax.set_title("LABS v1 · AUC-PR vs línea base (una barra verde por encima de la gris = señal real)")
ax.legend(); plt.tight_layout(); plt.savefig(FIG/"ap_vs_prevalencia_v1.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# CELDA 10 · EXPORTAR OOF/val/test PARA EL STACKING (hadm_id, labs_<label>, labs_<label>_cal)
def save_predictions(df, raw, cal, name):
    cols={"hadm_id":df["hadm_id"].to_numpy()}
    for j,l in enumerate(LABELS):
        key=l.replace(" ","_"); cols[f"labs_{key}"]=raw[:,j]; cols[f"labs_{key}_cal"]=cal[:,j]
    out=pd.DataFrame(cols); p=OUTPUT_DIR/f"labs_pred_{name}.csv"; out.to_csv(p,index=False); print("   guardado",p.name)
save_predictions(df_train, oof_train, oof_cal, "oof_train")
save_predictions(df_val, val_pred_raw, val_pred, "val")
save_predictions(df_test, test_pred_raw, test_pred, "test")
print("OOF/val/test del LABS v1 exportados.")

---
## ✅ Resumen — LABS v1 (ligero)
**Un único XGBoost** one-vs-rest con NaN nativo, ratios clínicos (BUN/Creatinina, Neutrófilos/Linfocitos, RDW×Edad) y
flags de missingness. Adopta el **etiquetado FINAL** (negativo = 0 + NaN→0; −1 enmascarado; **sin** derivar de
*No Finding*), la **métrica AUC-PR con IC bootstrap**, el balanceo por etiqueta, la calibración isotónica en VAL y el
**OOF sin fuga**. Para más rendimiento, ver **v2** (ensemble de 4 modelos) y **v3** (+TabPFN + stacking de 2º nivel).

### 📌 Para la documentación posterior (recuadros naranjas a redactar con los resultados)
- Comparar la **AP frente al AUC-ROC**: con desbalanceo la AP es más honesta y su línea base es la **prevalencia**
  de cada etiqueta, así que **no se comparan APs entre patologías** distintas.
- ¿Cuánto cambian las métricas respecto a la versión con negativos derivados de *No Finding*? Es el argumento que
  justifica el cambio de definición ante el tutor.
- Revisar la **importancia de los flags de missingness**: si dominan, el modelo puede estar usando el atajo espurio
  "tiene analíticas ⇒ está enfermo" en lugar de la biología (§6 EDA).
- Expectativa por etiqueta (§5/§9 EDA): el tabular debería aportar sobre todo en **Edema, Derrame y Cardiomegalia**,
  y quedarse cerca del azar en **Atelectasia y Opacidad** (esas las lidera la imagen).